# **Multi-Asset Portfolio Performance, Risk & Attribution**

**FINM3422 — Assessment 2 - Group 8**

*Analysis Period: January 2016 – December 2025*

# **1.0 Introduction**

introduction text

# **2.0 Data Overview**
### Environment Info, Imports and Config

In [2]:
import sys
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

sys.path.append("../src")
from data_loader import load_data, validate_data
from performance import all_sleeves_summary, annualised_return, annualised_volatility, max_drawdown
from attribution import brinson_attribution

print(sys.version)
print("pandas:", pd.__version__, "numpy:", np.__version__)
pd.set_option("display.float_format", "{:.4f}".format)

ImportError: cannot import name 'all_sleeves_summary' from 'performance' (/Users/ameliaperry/Desktop/BAFE/FINM3422/python case/FINM3422-Group-8-Task-2-1/notebook/../src/performance.py)

### Parameters and Paths

In [1]:
# Base paths
BASE_path = "/Users/ameliaperry/Desktop/BAFE/FINM3422/python case/FINM3422-Group-8-Task-2-1/data"

# TAA weights
taa_weights = pd.Series({
    "aus_eq": 0.35, 
    "intl_eq": 0.35, 
    "bonds": 0.15, 
    "re": 0.05, 
    "pevc": 0.10})

NameError: name 'pd' is not defined

### Data Load and Sanity Check

In [ ]:
# Load and Validate Data
df_managers, df_benchmarks, df_rf, df_saa = load_data(BASE_path)
validate_data(df_managers, df_benchmarks, df_rf)
saa_weights = df_saa["Weight"]

# Sanity Checks
print("Any nulls in benchmarks?", df_benchmarks.isna().sum().sum())
print("Any nulls in managers?", df_managers.isna().sum().sum())
print("Dates aligned?", df_benchmarks.index.equals(df_managers.index))
print("Monotonic dates?", df_benchmarks.index.is_monotonic_increasing)
display(df_managers.head())
display(df_benchmarks.head())
display(df_rf.head()) 
display(df_saa.head())

confirmation of data load and summary

# **3.0 Performance and Risk Analysis**

### Methodology

### Results Summary & Display

In [ ]:
summary = all_sleeves_summary(df_managers, df_benchmarks, df_rf)

pct_rows = [
    "Annualised Return (Manager)",
    "Annualised Return (Benchmark)",
    "Annualised Volatility",
    "Tracking Error",
    "Maximum Drawdown (Manager)",
    "Maximum Drawdown (Benchmark)",
]
ratio_rows = ["Sharpe Ratio", "Information Ratio"]

print("Performance & Risk Summary (%)")
display(summary.loc[pct_rows].T.map(lambda x: f"{x:.2%}"))

print("Ratios")
display(summary.loc[ratio_rows].T.map(lambda x: f"{x:.2f}"))

analysis

### Visualisations

In [ ]:
# Plots 1 to 5 - Cumulative Returns per sleeve
# manager vs benchmark
for sleeve in df_managers.columns:
    wealth_mgr = (1 + df_managers[sleeve]).cumprod()
    wealth_bm  = (1 + df_benchmarks[sleeve]).cumprod()
    plt.figure(figsize=(10, 4))
    plt.plot(wealth_mgr, label="Manager")
    plt.plot(wealth_bm, label="Benchmark")
    plt.title(sleeve.upper() + " Cumulative Returns")
    plt.xlabel("Date")
    plt.ylabel("Growth of $1")
    plt.legend()
    plt.grid(True)
    plt.show()

# Plot 6 - Sharpe Ratios across all sleeves
sleeves = list(df_managers.columns)
sharpe_ratios = [summary.loc["Sharpe Ratio", sleeve] for sleeve in sleeves]

plt.figure(figsize=(8, 4))
plt.bar(sleeves, sharpe_ratios)
plt.title("Sharpe Ratios by Sleeve")
plt.ylabel("Sharpe Ratio")
plt.show()

#Plot 7 - Information Ratios across all sleeves
sleeves = list(df_managers.columns)
ir_ratios = [summary.loc["Information Ratio", sleeve] for sleeve in sleeves]

plt.figure(figsize=(8, 4))
plt.bar(sleeves, ir_ratios)
plt.title("Information Ratios by Sleeve")
plt.ylabel("Information Ratio")
plt.show()

analysis

# **4.0 APRA-Inspired Performance and Risk Checks**

### TAA Weights and Total Fund Return

In [ ]:
#Total Fund Monthly Returns
# weighted average of manager returns using TAA weights
df_fund_monthly = df_managers.mul(taa_weights).sum(axis=1)

# Total Benchmark Monthly Returns
# weighted average of benchmark returns using SAA weights
saa_weights = df_saa["Weight"]
df_benchmark_monthly = df_benchmarks.mul(saa_weights, axis=1).sum(axis=1)

print("Fund monthly returns (first 5):")
display(df_fund_monthly.head())

### APRA Checks

In [ ]:
# Inputs
CPI_TARGET = 0.025 # assumed CPI of 2.5%
RETURN_PREMIUM = 0.035 # difference between CPI and return target
RETURN_TARGET = CPI_TARGET + RETURN_PREMIUM # 6% return target
VOL_LIMIT = 0.10 # 10% annualised volatility limit
DRAWDOWN_LIMIT = -0.25 # -25% maximum drawdown limit

# Fund Level Metrics
fund_ann_return = annualised_return(df_fund_monthly)
fund_ann_vol = annualised_volatility(df_fund_monthly)
fund_mdd = max_drawdown(df_fund_monthly)

# Check 1: Long-term return vs target
check1_pass = fund_ann_return >= RETURN_TARGET

# Check 2: Max Drawdown vs -25% Limit
check2_pass = fund_mdd >= DRAWDOWN_LIMIT

# Check 3: Volatility vs 10% Limit
check3_pass = fund_ann_vol <= VOL_LIMIT

# Results
APRA_results = pd.DataFrame({
    "Check": ["Long-run return >= CPI + 3.5% (6.0%)", "Max Drawdown >= -25%", "Annualised Volatility <= 10%"],
    "Fund Value": [f"{fund_ann_return:.2%}", f"{fund_mdd:.2%}", f"{fund_ann_vol:.2%}"],
    "Threshold": [f"{RETURN_TARGET:.2%}",f"{DRAWDOWN_LIMIT:.2%}", f"{VOL_LIMIT:.2%}"],
    "Pass / Fail": ["Pass" if check1_pass else "Fail", "Pass" if check2_pass else "Fail", "Pass" if check3_pass else "Fail"]})

print("APRA Performance & Risk Checks:")
display(APRA_results)

analysis

### Shock Scenario A

20% Equity Crash in one month

In [ ]:
# Apply shock to returns in month recent months
shock_returns_a = df_managers.iloc[-1].copy() # use last month returns as base
shock_returns_a["aus_eq"] = -0.20 # apply -20% shock to Australian equity
shock_returns_a["intl_eq"] = -0.20 # apply -20% shock to International equity
# Bonds, RE, PEVC assumed unaffected in shock scenario (equities only)

# Calculate shocked fund return for that month
return_shock_a = (taa_weights * shock_returns_a).sum()

# Comparison to normal return
return_normal_a = (taa_weights * df_managers.iloc[-1]).sum()

print("Shock Scenario: Equity Crash (-20% AUS EQ & INTL EQ in one month)")
shock_scenario_a_results = pd.DataFrame({
    "Scenario": ["Normal", "Equity Crash Shock"],
    "Portfolio Return": [f"{return_normal_a:.2%}", f"{return_shock_a:.2%}"],
    "AUS EQ Return": [f"{df_managers.iloc[-1]['aus_eq']:.2%}", "-20.00%"],
    "INTL EQ Return": [f"{df_managers.iloc[-1]['intl_eq']:.2%}", "-20.00%"]})
display(shock_scenario_a_results)
print(f"\nShock Impact on Portfolio Returns: {return_shock_a - return_normal_a:.2%}")

analysis

### Shock Scenario B - Bond Yield Spike

bond yields inrase 150bps

In [ ]:
# bonds: -(duration x 1.5%) = -5%
# REITS: -5%
# Equities: -2%
# PE/VC: -2%

shock_returns_b = df_managers.iloc[-1].copy()
shock_returns_b["aus_eq"] = -0.02
shock_returns_b["intl_eq"] = -0.02
shock_returns_b["bonds"] = -0.05
shock_returns_b["re"] = -0.05
shock_returns_b["pevc"] = -0.02

return_shock_b = (taa_weights * shock_returns_b).sum()
return_normal_b = (taa_weights * df_managers.iloc[-1]).sum()

print("Shock Scenario B: Bond Yield Spike")
shock_scenario_b_results = pd.DataFrame({
    "Scenario": ["Normal (last month)", "Bond Yield Spike"],
    "AUS EQ": [f"{df_managers.iloc[-1]['aus_eq']:.2%}", "-2.00%"],
    "INTL EQ": [f"{df_managers.iloc[-1]['intl_eq']:.2%}", "-2.00%"],
    "Bonds": [f"{df_managers.iloc[-1]['bonds']:.2%}", "-5.00%"],
    "RE": [f"{df_managers.iloc[-1]['re']:.2%}", "-5.00%"],
    "PEVC": [f"{df_managers.iloc[-1]['pevc']:.2%}", "-2.00%"],
    "Portfolio Return": [f"{return_normal_b:.2%}", f"{return_shock_b:.2%}"]})
display(shock_scenario_b_results)
print(f"\nShock Impact on Portfolio Returns: {return_shock_b - return_normal_b:.2%}")

analysis

### Total Fund Vs. Benchmark Comparison

In [ ]:
# Summary Comparison Table
fund_vol = annualised_volatility(df_fund_monthly)
bm_return = annualised_return(df_benchmark_monthly)
bm_vol = annualised_volatility(df_benchmark_monthly)
bm_mdd = max_drawdown(df_benchmark_monthly)

comparison = pd.DataFrame({
    "Metric": ["Annualised Return", "Annualised Volatility", "Maximum Drawdown"],
    "Total Fund (TAA)": [f"{fund_ann_return:.2%}", f"{fund_vol:.2%}", f"{fund_mdd:.2%}"],
    "Composite Benchmark (SAA)": [f"{bm_return:.2%}", f"{bm_vol:.2%}", f"{bm_mdd:.2%}"]
})
print("Total Fund vs Composite Benchmark Performance:")
display(comparison)

analysis

# **5.0 Multi-Asset Performance Attribution**

### Formulas and Methodology

### Imports and Attribution

analysis

analysis

### Verify Active Return

analysis

# **6.0 Conclusion**

conclusion text